# 05 • Diagnostic et régularisation

`[MÉTA | Formation 4-024 | Niveau Application | TP 05 | Mode CPU local]`

**Objectif :** Interpréter les courbes et tester une hypothèse isolée.

**Temps indicatif :** Atelier diagnostic J2. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** TP 02 et 04.

**Preuves de réussite :** Micro-lot mémorisé, comparaison tracée, effet train/eval observé.

**Sources :** R02, R10 à R12.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [1]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


Moteur disponible : 2.10.0+cpu | Données : /mnt/data/deep_learning_4_024/03_Travaux_pratiques/donnees


## 1. Test du micro-lot
Un petit réseau suffisamment capable doit pouvoir mémoriser 32 exemples cohérents. Ce test recherche un problème de boucle, pas la généralisation. La performance sur ces 32 exemples ne doit jamais être présentée comme une performance finale.

In [2]:
d=digits_data();tiny=(d['train'][0][:32],d['train'][1][:32])
seed_all();m=DenseDigits();hist=fit_model(m,tiny,tiny,task='multi',epochs=160,lr=.01,batch_size=32)
p=predict(m,tiny[0],'multi').argmax(1);acc=float(accuracy_score(tiny[1],p))
print('Mémorisation micro-lot :',acc)
assert acc>=.95,'Vérifier la boucle et augmenter le nombre de pas avant de poursuivre.'
plot_history(hist,'Micro-lot : test de fonctionnement, pas généralisation','05_micro_lot.png')

Mémorisation micro-lot : 1.0


<Figure size 700x400 with 1 Axes>

## 2. Expériences contrôlées
Sur un sous-ensemble d’entraînement réduit, comparer référence, dropout seul et pénalisation L2 via weight_decay seul. Les initialisations sont réinitialisées. Le test reste fermé. Aucun résultat n’est promis ; les courbes décident de l’interprétation.

In [3]:
small=(d['train'][0][:200],d['train'][1][:200]);reports={}
for name,drop,decay in [('reference',0.,0.),('dropout',.4,0.),('penalisation',0.,.005)]:
    seed_all();m=TinyCNN(dropout=drop)
    h=fit_model(m,small,d['validation'],task='multi',epochs=25,weight_decay=decay)
    reports[name]={'validation_exactitude':float(accuracy_score(d['validation'][1],predict(m,d['validation'][0],'multi').argmax(1))),'derniere_perte_train':float(h.iloc[-1].perte_train),'derniere_perte_validation':float(h.iloc[-1].perte_validation)}
    plot_history(h,name,'05_'+name+'.png')
print(pd.DataFrame(reports).T)
save_result('05_regularisation',reports)

              validation_exactitude  ...  derniere_perte_validation
reference                  0.818519  ...                   0.648863
dropout                    0.737037  ...                   1.056823
penalisation               0.800000  ...                   0.744496

[3 rows x 3 columns]


PosixPath('/mnt/data/deep_learning_4_024/03_Travaux_pratiques/resultats/05_regularisation.json')

## 3. Dropout : entraînement et inférence
Deux appels en mode entraînement peuvent différer. Deux appels en mode évaluation, sans autre source d’aléa, doivent coïncider. La correction n’est pas « supprimer dropout », mais utiliser le mode adapté.

In [4]:
couche=nn.Dropout(.5);x=torch.ones(128)
couche.train();a=couche(x);b=couche(x)
couche.eval();u=couche(x);v=couche(x)
assert not torch.equal(a,b) and torch.equal(u,v)
print('Train identique ?',torch.equal(a,b),'Évaluation identique ?',torch.equal(u,v))

Train identique ? False Évaluation identique ? True


## 4. BatchNorm : état courant et statistiques mémorisées
BatchNorm ne se réduit pas à changer la moyenne des données d’entrée. Son comportement dépend du mode. Ici, on observe une moyenne mobile interne après un lot. Les petits lots et les distributions très différentes peuvent compliquer l’usage.

In [5]:
bn=nn.BatchNorm1d(3);avant=bn.running_mean.clone();bn.train()
bn(torch.randn(16,3)+5);apres=bn.running_mean.clone()
print('Moyenne mémorisée avant :',avant,'après :',apres)
assert not torch.equal(avant,apres)
bn.eval();etat=bn.running_mean.clone();bn(torch.randn(16,3)+50)
assert torch.equal(etat,bn.running_mean)

Moyenne mémorisée avant : tensor([0., 0., 0.]) après : tensor([0.4746, 0.5152, 0.4598])


## 5. Journal demandé
Hypothèse, symptôme, changement unique, métrique attendue, résultat obtenu, autre explication possible, décision suivante. **Remédiation :** si train et validation sont mauvais, ne pas augmenter aveuglément la régularisation. Inspecter d’abord la boucle et les données. **Extension :** mettre en œuvre un arrêt anticipé avec patience=4 et conserver le meilleur état de validation.